# 32 · The paper story: where human and mouse PT differ, and which genes carry it

Notebook 31 chose the analysis and froze a list of 82 confident pathways. This notebook does not
test anything new. It selects readable examples from that frozen list, shows the genes behind
them, and makes the contrast with conventional analysis concrete.

**The claim this supports.** Human and mouse proximal tubule differ less in *whether* a metabolic
program is present than in *where along the tubule axis* it is deployed. A whole-PT or
S1/S2/S3 comparison reports one number per gene per region and cannot express that. The
continuous species-by-position model can, and notebook 31 showed it is the only screen here whose
discoveries do not reappear when specimens are relabeled.

Cohort reminder: 2 control mice, 2 cortex sections from 1 male human donor, all specimens male.
Descriptive throughout.

In [ ]:
import argparse
import json
import os
import re
import sys
from pathlib import Path

start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
upstream13 = results_root / 'minimal_pt_scfates'
upstream31 = results_root / 'pt_pathway_method_selection'
output = results_root / 'pt_pathway_story'
figures = output / 'figures'
figures.mkdir(parents=True, exist_ok=True)
for path in (upstream31 / 'final_pathway_table.csv', upstream31 / 'gene_positional_summary.csv',
             upstream31 / 'confident_pathway_divergence_curves.csv', upstream31 / 'run_manifest.json'):
    if not path.is_file():
        raise FileNotFoundError(f'Run notebook 31 first: missing {path}')

STORY_LOGIC_VERSION = '32.pathway_story.1'
BASIS_DF = 6
print('Results folder:', output)

## 1 · Rebuild the same structures, genes and fitted curves

Identical to notebooks 12 and 31: same PT structures, common coordinate support, count-based
normalization, 2% detection rule. The fitted model is checked against notebook 12's saved
statistics before anything is plotted.

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse
from IPython.display import display
from pseudospace.pathway_inputs import rebuild_pt_expression
from pseudospace.pathway_remodeling import fit_nested_trajectories, gene_covariates, local_pathway_curves
from pseudospace.pathways import build_pathway_membership
from pseudospace.stage_cache import cached_payload, digest
import pseudospace.pathway_remodeling as remodeling

saved_pt = ad.read_h5ad(upstream13 / 'cross_species_pt_scfates.h5ad', backed='r')
pt_obs = saved_pt.obs.copy()
saved_pt.file.close()
pt_obs = pt_obs[pt_obs.broad_tubule_marker_call.astype(str).eq('PT')].copy()
expected = {'mouse': ['Ctrl1A2', 'Ctrl1A4'], 'human': ['HUK1_COR1', 'HUK1_MED1']}
support = pt_obs.groupby('sample', observed=True).shared_pseudospace.quantile([.01, .99]).unstack()
coordinate = pt_obs.shared_pseudospace.to_numpy(float)
pt_obs = pt_obs[(coordinate >= support[.01].max()) & (coordinate <= support[.99].min())].copy()
orthologs = pd.read_csv(upstream13 / 'ortholog_map_used.csv')
adata = rebuild_pt_expression(pt_obs, {n: data_root / 'tubule_by_gene' / f'{n}_tubule_by_gene_caleb.h5ad'
                                       for v in expected.values() for n in v}, orthologs, target_sum=1e4)
position = adata.obs.shared_pseudospace.to_numpy(float)
human = adata.obs.comparison_species.astype(str).eq('human').to_numpy()
specimen = adata.obs['sample'].astype(str).to_numpy()
segment = adata.obs.segment_class.astype(str).to_numpy()
names = expected['mouse'] + expected['human']
lo = max(position[specimen == n].min() for n in names)
hi = min(position[specimen == n].max() for n in names)
grid = np.linspace(lo, hi, 61)
expression = sparse.csr_matrix(adata.layers['lognorm'], dtype=float)
universe = gene_covariates(expression, position, human, specimen, adata.var_names)
universe['eligible'] = adata.var.measured_in_both_inputs.to_numpy() & universe.detection.ge(.02)
genes = universe.loc[universe.eligible, 'gene'].tolist()
gene_index = pd.Index(genes)
Y = expression[:, universe.eligible.to_numpy()]
coverage = pd.concat([build_pathway_membership(
    json.loads((data_root / 'mouse_vs_human' / 'pathway_gene_sets' / f'{library}.json').read_text()),
    universe.loc[adata.var.measured_in_both_inputs.to_numpy(), 'gene'], ortholog_map=orthologs,
    library_name=library, tested=genes, min_genes=1)
    for library in ('Reactome_2022', 'MSigDB_Hallmark_2020', 'KEGG_2019_Mouse')], ignore_index=True)
coverage['pathway_id'] = coverage.library + '::' + coverage.pathway
coverage = coverage[coverage.n_tested.between(10, min(300, len(genes) - 1))]
gene_sets = {row.pathway_id: list(row.genes_present) for row in coverage.itertuples()}

fit = cached_payload('story_nested_fit',
    lambda: dict(fit_nested_trajectories(Y, position, human, specimen, grid, basis_df=BASIS_DF)),
    root=output / 'stage_cache', params={'basis_df': BASIS_DF, 'grid': grid},
    inputs={'Y': digest(Y), 'position': position, 'human': human, 'specimen': specimen, 'genes': genes},
    code=digest([STORY_LOGIC_VERSION, Path(remodeling.__file__)]))
saved_gene_stats = pd.read_csv(results_root / 'pt_pathway_remodeling_scfates' / 'gene_statistics.csv', index_col=0)
np.testing.assert_allclose(fit['T_spatial'], saved_gene_stats.T_spatial, rtol=1e-6, atol=1e-8)
delta = pd.DataFrame(fit['delta'], index=genes, columns=np.round(grid, 4))   # human − mouse, log-norm
zscore = pd.DataFrame(fit['z'], index=genes, columns=np.round(grid, 4))
human_curve = pd.DataFrame(fit['human'], index=genes, columns=np.round(grid, 4))
mouse_curve = pd.DataFrame(fit['mouse'], index=genes, columns=np.round(grid, 4))
print(f'{len(position):,} structures · {len(genes):,} genes · fitted curves reproduce notebook 12')

## 2 · The frozen list and how examples are chosen

We take notebook 31's 82 confident pathways and keep only those it classed **new with
pseudospace** (no conventional whole-PT or segment species call) and retained in **all seven**
planned sensitivity runs. From that set we choose five examples spanning distinct themes,
preferring large spatial effect and the presence of direction-reversing member genes. These are
readability choices among already-frozen results; they add no discovery and change no statistic.

In [ ]:
final = pd.read_csv(upstream31 / 'final_pathway_table.csv')
gene_table = pd.read_csv(upstream31 / 'gene_positional_summary.csv', index_col=0)
confident = final[final.confident.astype(bool)].copy()
eligible = confident[confident.pseudospace_class.eq('new with pseudospace')
                     & confident.retention_fraction.ge(.999)].copy()
FLAGSHIPS = ['Reactome_2022::Drug ADME R-HSA-9748784',
             'Reactome_2022::Mitochondrial Fatty Acid Beta-Oxidation R-HSA-77289',
             'KEGG_2019_Mouse::Valine, leucine and isoleucine degradation',
             'KEGG_2019_Mouse::Glutathione metabolism',
             'Reactome_2022::Retinoid Metabolism And Transport R-HSA-975634']
missing = [p for p in FLAGSHIPS if p not in set(eligible.pathway_id)]
if missing:
    raise ValueError(f'A chosen example is no longer eligible; revisit the selection: {missing}')
flagships = eligible.set_index('pathway_id').loc[FLAGSHIPS].reset_index()
display(flagships[['name', 'theme', 'n_tested', 'effect_T_spatial', 'q_spatial', 'q_level',
                   'peak_position', 'affected_width', 'reversing_members']].round(3))
print(f'{len(eligible)} eligible examples; {len(FLAGSHIPS)} shown.')

## 3 · Figure 5 · Each example, four ways of looking at it

| Panel | What it shows |
|---|---|
| Local divergence D(s) | Where the pathway's genes separate from the background along pseudospace |
| Member genes | Fitted human − mouse difference per gene, ordered by where each gene peaks |
| Fitted curves | Human and mouse expression for two genes, with each specimen's binned means as dots |
| Conventional view | Whole-PT and per-segment DESeq2 log2 fold change for the same two genes |

The conventional panel is the comparison the paper makes: one number per gene, or three, against
a curve. Shaded bands mark where the reviewed S1/S2/S3 clusters sit on the coordinate.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 9,
                     'axes.titlesize': 10, 'axes.titleweight': 'bold', 'svg.fonttype': 'none',
                     'pdf.fonttype': 42})
HUMAN, MOUSE, INK = '#D55E00', '#0072B2', '#333333'
segments = ('PT-S1', 'PT-S2', 'PT-S3')
segment_spans = {s: np.quantile(position[segment == s], [.1, .9]) for s in segments}
edges = np.linspace(lo, hi, 13)
centres = (edges[:-1] + edges[1:]) / 2
binned = np.clip(np.searchsorted(edges, position, side='right') - 1, 0, len(centres) - 1)
conventional = pd.read_csv(upstream31 / 'conventional_deseq2_all_partitions.csv')
conventional = conventional[conventional.partition.eq('species')]
lfc = conventional.pivot(index='gene', columns='scope', values='log2FoldChange')
padj = conventional.pivot(index='gene', columns='scope', values='padj')
curves = local_pathway_curves(fit['z'], genes, {p: gene_sets[p] for p in FLAGSHIPS}, grid)


def save(fig, name):
    for suffix in ('pdf', 'png'):
        fig.savefig(figures / f'{name}.{suffix}', bbox_inches='tight', dpi=200)


def shade(ax):
    for s, (a, b) in segment_spans.items():
        ax.axvspan(a, b, color='#000000', alpha=.04 + .03 * segments.index(s), lw=0, zorder=0)


def specimen_points(gene):
    column = Y[:, gene_index.get_loc(gene)].toarray().ravel()
    return {name: np.array([column[(specimen == name) & (binned == b)].mean()
                            if np.any((specimen == name) & (binned == b)) else np.nan
                            for b in range(len(centres))]) for name in names}


def example_genes(pathway_id, n=2):
    """The member genes whose species difference changes most along the PT."""
    members = [g for g in gene_sets[pathway_id] if g in gene_table.index]
    table = gene_table.loc[members]
    table = table[table[['detection_mouse', 'detection_human']].max(axis=1).ge(.10)]
    return table.sort_values('shape_amplitude', ascending=False).head(n).index.tolist()


story_rows = []
for row in flagships.itertuples():
    members = [g for g in gene_sets[row.pathway_id] if g in gene_table.index]
    chosen = example_genes(row.pathway_id)
    block = delta.loc[members]
    block = block.loc[block.abs().idxmax(axis=1).sort_values().index]
    fig = plt.figure(figsize=(12.2, 3.5))
    gs = fig.add_gridspec(1, 4, width_ratios=[1, .95, 1.15, .75], left=.055, right=.985, top=.78,
                          bottom=.17, wspace=.55)
    ax = fig.add_subplot(gs[0])
    shade(ax)
    c = curves[curves.pathway_id.eq(row.pathway_id)]
    ax.plot(c.position, c.divergence, color=INK, lw=1.8)
    ax.axhline(0, color='#999999', lw=.7)
    ax.axvline(row.peak_position, color=HUMAN, lw=1, ls='--')
    for s, (a, b) in segment_spans.items():
        ax.text((a + b) / 2, 1.01, s.replace('PT-', ''), transform=ax.get_xaxis_transform(),
                ha='center', va='bottom', fontsize=7.5, color='#555555')
    ax.set(xlabel='PT pseudospace', ylabel='Local divergence D(s)', xlim=(lo, hi))
    ax.set_title('Where it diverges', loc='left', fontsize=9, pad=14)
    ax = fig.add_subplot(gs[1])
    limit = max(.5, float(np.quantile(np.abs(block.to_numpy()), .95)))
    im = ax.imshow(block.clip(-limit, limit).to_numpy(), aspect='auto', cmap='RdBu_r',
                   vmin=-limit, vmax=limit, extent=(lo, hi, len(block), 0), interpolation='nearest')
    show = [g if g in chosen else '' for g in block.index]
    ax.set_yticks(np.arange(len(block)) + .5, show, fontsize=7.5)
    ax.set(xlabel='PT pseudospace')
    ax.set_title(f'{len(block)} member genes', loc='left', fontsize=9)
    cax = ax.inset_axes([1.04, .1, .035, .8])
    fig.colorbar(im, cax=cax)
    cax.set_title('human\n− mouse', fontsize=7, loc='left')
    inner = gs[2].subgridspec(len(chosen), 1, hspace=.5)
    for i, gene in enumerate(chosen):
        sub = fig.add_subplot(inner[i])
        shade(sub)
        sub.plot(grid, human_curve.loc[gene], color=HUMAN, lw=2)
        sub.plot(grid, mouse_curve.loc[gene], color=MOUSE, lw=2)
        for name, profile in specimen_points(gene).items():
            sub.scatter(centres, profile, s=7, color=HUMAN if name in expected['human'] else MOUSE,
                        alpha=.75, zorder=3)
        sub.set_ylabel(gene, fontsize=8.5, fontweight='bold')
        sub.set_xlim(lo, hi)
        sub.tick_params(labelsize=7.5)
        if i == 0:
            sub.set_title('Fitted curves (dots: specimen means)', loc='left', fontsize=9)
        if i == len(chosen) - 1:
            sub.set_xlabel('PT pseudospace')
        else:
            sub.set_xticklabels([])
    inner = gs[3].subgridspec(len(chosen), 1, hspace=.5)
    for i, gene in enumerate(chosen):
        sub = fig.add_subplot(inner[i])
        values = lfc.loc[gene, ['whole_PT', *segments]]
        bars = sub.bar(range(4), values.to_numpy(),
                       color=['#555555'] + [HUMAN if v > 0 else MOUSE for v in values[1:]], width=.7)
        for j, gene_padj in enumerate(padj.loc[gene, ['whole_PT', *segments]]):
            if pd.notna(gene_padj) and gene_padj <= .05:
                sub.text(j, values.iloc[j], '*', ha='center',
                         va='bottom' if values.iloc[j] > 0 else 'top', fontsize=9)
        sub.axhline(0, color='#999999', lw=.6)
        sub.set_xticks(range(4), ['PT', 'S1', 'S2', 'S3'] if i == len(chosen) - 1 else [''] * 4,
                       fontsize=7.5)
        sub.tick_params(labelsize=7.5)
        if i == 0:
            sub.set_title('Conventional log2FC', loc='left', fontsize=9)
        story_rows.append({'pathway': row.name, 'gene': gene,
                           **{f'lfc_{k}': lfc.loc[gene, k] for k in ['whole_PT', *segments]},
                           **{f'padj_{k}': padj.loc[gene, k] for k in ['whole_PT', *segments]},
                           **gene_table.loc[gene, ['T_spatial', 'shape_amplitude', 'peak_delta',
                                                   'peak_delta_position', 'delta_mean',
                                                   'reverses_along_PT']].to_dict()})
    fig.suptitle(f'{row.name}   ·   {row.theme}   ·   spatial q = {row.q_spatial:.3f}, '
                 f'offset q = {row.q_level:.2f}', x=.055, ha='left', fontsize=10.5)
    save(fig, 'fig5_example_' + re.sub(r'[^a-z0-9]+', '_', row.name.lower()).strip('_'))
    plt.show()
    plt.close(fig)
pd.DataFrame(story_rows).to_csv(output / 'example_gene_evidence.csv', index=False)

## 4 · Figure 6 · Genes that change direction along the tubule

A whole-PT comparison assigns one direction per gene. 70 genes have a fitted species difference
that is clearly human-high in one part of the PT and clearly mouse-high in another (|Z| ≥ 3 and
|difference| ≥ 0.25 log-normalized units over at least three contiguous grid points on each
side). For these genes the conventional answer is not merely less precise; it reports a single
sign for a quantity that has two. The bars give the whole-PT log2 fold change for the same genes.

In [ ]:
reversing = gene_table[gene_table.reverses_along_PT.astype(bool)].copy()
reversing['in_confident_pathway'] = [any(g in gene_sets[p] for p in confident.pathway_id)
                                     for g in reversing.index]
def crossing(values):
    """Zero crossing between the largest human-high and mouse-high points, by linear interpolation."""
    a, b = sorted((int(np.argmax(values)), int(np.argmin(values))))
    changes = [i for i in range(a, b) if np.sign(values[i]) != np.sign(values[i + 1])]
    i = max(changes, key=lambda k: abs(values[k + 1] - values[k]))
    return float(grid[i] + (grid[i + 1] - grid[i]) * values[i] / (values[i] - values[i + 1]))


reversing['crossing'] = [crossing(delta.loc[g].to_numpy()) for g in reversing.index]
reversing = reversing.sort_values('crossing')
reversing.to_csv(output / 'direction_reversing_genes.csv')
show = reversing[reversing.in_confident_pathway].sort_values('shape_amplitude', ascending=False).head(24)
show = show.sort_values('crossing')
fig = plt.figure(figsize=(9.6, 6.4))
gs = fig.add_gridspec(1, 3, width_ratios=[1.5, .5, .45], left=.12, right=.97, top=.9, bottom=.09, wspace=.1)
ax = fig.add_subplot(gs[0])
block = delta.loc[show.index]
limit = float(np.quantile(np.abs(block.to_numpy()), .98))
im = ax.imshow(block.clip(-limit, limit).to_numpy(), aspect='auto', cmap='RdBu_r', vmin=-limit, vmax=limit,
               extent=(lo, hi, len(block), 0), interpolation='nearest')
ax.scatter(show.crossing, np.arange(len(show)) + .5, marker='|', s=40, color='black', lw=1.2)
ax.set_yticks(np.arange(len(show)) + .5, show.index, fontsize=8)
ax.set_xlabel('PT pseudospace')
ax.set_title('Fitted species difference reverses along the PT', loc='left')
heat_image = im
ax = fig.add_subplot(gs[1])
values = lfc.reindex(show.index).whole_PT
ax.barh(np.arange(len(show))[::-1], values.to_numpy(),
        color=[HUMAN if v > 0 else MOUSE for v in values.fillna(0)], height=.65)
ax.axvline(0, color='#999999', lw=.7)
ax.set_yticks([])
ax.set_ylim(-.5, len(show) - .5)
ax.set_xlabel('Whole-PT log2FC')
ax.set_title('What whole-PT reports', loc='left', fontsize=9)
ax = fig.add_subplot(gs[2])
ax.axis('off')
ax.text(0, 1, 'The heatmap row changes colour.\nThe bar beside it cannot.\n\n'
        f'{int(reversing.in_confident_pathway.sum())} of {len(reversing)} reversing genes\n'
        'belong to a confident pathway.\n\nTick = where the fitted\ndifference crosses zero.',
        va='top', fontsize=8.5)
cax = ax.inset_axes([0, .42, .9, .025])
fig.colorbar(heat_image, cax=cax, orientation='horizontal')
cax.set_title('human − mouse (log-norm)', fontsize=8, loc='left')
save(fig, 'fig6_direction_reversing_genes')
plt.show()
plt.close(fig)

## 5 · Figure 7 · The summary claim, in one panel

For every one of the 82 confident pathways, the position of its largest divergence against its
spatial effect size. Colour is the direction of the median member difference at that position.
The point is the spread along the horizontal axis: these pathways do not all differ in the same
place, which is what a whole-PT or three-segment summary assumes.

In [ ]:
summary = confident.copy()
summary['direction'] = np.where(summary.median_member_z_at_peak > 0, 'human-high at peak', 'mouse-high at peak')
fig, (ax, side) = plt.subplots(1, 2, figsize=(9.4, 4.4), gridspec_kw={'width_ratios': [3, 1], 'wspace': .25},
                               layout='constrained')
shade(ax)
for label, color, marker in (('human-high at peak', HUMAN, 'o'), ('mouse-high at peak', MOUSE, 's')):
    part = summary[summary.direction.eq(label)]
    ax.scatter(part.peak_position, part.effect_T_spatial, s=26 + 90 * part.affected_width,
               color=color, marker=marker, alpha=.75, edgecolor='white', lw=.6, label=label)
for s, (a, b) in segment_spans.items():
    ax.text((a + b) / 2, 1.01, s.replace('PT-', ''), transform=ax.get_xaxis_transform(), ha='center',
            va='bottom', fontsize=8, color='#555555')
ax.set(xlabel='Position of largest divergence', ylabel='Spatial enrichment effect (AUC − 0.5)', xlim=(lo, hi))
ax.legend(frameon=False, fontsize=8, loc='upper center', bbox_to_anchor=(.5, -.13), ncol=2)
ax.set_title('82 confident pathways differ in different places', loc='left', pad=16)
counts = summary.groupby('theme').size().sort_values()
side.barh(np.arange(len(counts)), counts.to_numpy(), color='#BBBBBB', height=.65)
new_counts = summary[summary.pseudospace_class.eq('new with pseudospace')].groupby('theme').size().reindex(counts.index).fillna(0)
side.barh(np.arange(len(counts)), new_counts.to_numpy(), color=HUMAN, height=.65)
side.set_yticks(np.arange(len(counts)), [t.replace(' & ', ' &\n') for t in counts.index], fontsize=7.5)
side.set_xlabel('Pathways')
side.set_title('Orange: new with\npseudospace', loc='left', fontsize=9)
save(fig, 'fig7_confident_pathway_landscape')
plt.show()
plt.close(fig)

## 6 · Export

In [ ]:
export = confident[['pathway_id', 'library', 'name', 'theme', 'n_tested', 'effect_T_spatial', 'q_spatial',
                    'q_level', 'peak_position', 'affected_width', 'median_member_z_at_peak',
                    'reversing_members', 'pseudospace_class', 'retention_fraction', 'q_corr']].copy()
export['flagship'] = export.pathway_id.isin(FLAGSHIPS)
export.sort_values(['flagship', 'q_spatial'], ascending=[False, True]).to_csv(
    output / 'confident_pathways_for_paper.csv', index=False)
member_rows = []
for row in flagships.itertuples():
    for gene in gene_sets[row.pathway_id]:
        if gene in gene_table.index:
            member_rows.append({'pathway': row.name, 'gene': gene,
                                **gene_table.loc[gene, ['T_spatial', 'shape_amplitude', 'peak_delta',
                                                        'peak_delta_position', 'delta_mean',
                                                        'reverses_along_PT', 'detection_mouse',
                                                        'detection_human']].to_dict(),
                                'lfc_whole_PT': lfc.get('whole_PT', pd.Series()).get(gene, np.nan)})
pd.DataFrame(member_rows).to_csv(output / 'flagship_member_genes.csv', index=False)
manifest = {'story_logic_version': STORY_LOGIC_VERSION, 'upstream': '31.method_selection.1',
            'flagships': FLAGSHIPS, 'n_confident': int(len(confident)),
            'n_eligible_examples': int(len(eligible)),
            'n_reversing_genes': int(len(reversing)), 'n_genes': len(genes),
            'caveat': 'Examples selected for readability from a frozen list; no new test. Two mice and '
                      'two cortex sections from one male human donor.'}
(output / 'run_manifest.json').write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))